# Batch Section-Based Extraction Evaluation

Evaluate section-based metadata extraction using GROBID-parsed PDFs across all Fuster validation samples.

**Approach:**
- Parse PDFs with GROBID to extract structured sections
- Select relevant sections (Methods, Data, Study Area, etc.) using section classification
- Build full-text prompts from selected sections
- Run GPT-5-mini extraction with section-aware system prompt
- Compare against manually annotated ground truth
- Use `DatasetFeaturesNormalized` for ground truth validation

In [1]:
print("hello Vincent")

%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

# Project modules - using section_pipeline (replaces deprecated fulltext_pipeline)
from llm_metadata.section_pipeline import (
    SectionClassificationConfig,
    SectionSelectionConfig,
    SectionInputRecord,
    SectionOutputRecord,
    # Flows
    section_classification_flow,
    single_section_classification,
    # Section selection utilities
    collect_relevant_sections,
    build_fulltext_prompt,
    # Manifest I/O
    load_section_manifest,
    save_section_manifest,
)
from llm_metadata.pdf_parsing import process_pdf, ParsedDocument, Section
from llm_metadata.gpt_classify import classify_abstract, SECTION_SYSTEM_MESSAGE
from llm_metadata.chunking import count_tokens
from llm_metadata.section_normalize import classify_section
from llm_metadata.schemas.chunk_metadata import SectionType
from llm_metadata.schemas.fuster_features import (
    DatasetFeatures,
    DatasetFeaturesNormalized,
)
from llm_metadata.groundtruth_eval import (
    evaluate_indexed,
    EvaluationConfig,
    FuzzyMatchConfig,
    micro_average,
    macro_f1,
)

# Paths
PDF_DIR = Path("data/pdfs/fuster")
MANIFEST_PATH = PDF_DIR / "manifest.csv"
GROUND_TRUTH_PATH = Path("data/dataset_092624.xlsx")
OUTPUT_DIR = Path("artifacts/section_results")

print(f"PDF directory: {PDF_DIR.absolute()}")
print(f"Manifest exists: {MANIFEST_PATH.exists()}")
print(f"Ground truth exists: {GROUND_TRUTH_PATH.exists()}")

hello Vincent


PDF directory: C:\Users\beav3503\dev\llm_metadata\notebooks\data\pdfs\fuster
Manifest exists: False
Ground truth exists: False


C:\Users\beav3503\AppData\Local\miniforge3\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value <class 'llm_metadata.schemas.abstract_metadata.DatasetAbstractMetadata'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
C:\Users\beav3503\AppData\Local\miniforge3\Lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value <class 'llm_metadata.schemas.abstract_metadata.DatasetAbstractMetadata'> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


## Step 1: Load Manifest and Ground Truth

Map dataset DOIs to article DOIs and load ground truth annotations.

In [16]:
# Load manifest with DOI mapping
manifest_df = pd.read_csv(MANIFEST_PATH)
print(f"Manifest: {len(manifest_df)} total rows")

# Filter to downloaded PDFs only
downloaded_df = manifest_df[manifest_df['status'] == 'downloaded'].copy()
print(f"Downloaded PDFs: {len(downloaded_df)}")

# Create DOI mapping (dataset_doi -> article_doi)
doi_mapping = dict(zip(downloaded_df['dataset_doi'], downloaded_df['article_doi']))
print(f"DOI mappings: {len(doi_mapping)}")

# Display sample
downloaded_df[['article_doi', 'dataset_doi', 'downloaded_pdf_path']].head(5)

Manifest: 75 total rows
Downloaded PDFs: 70
DOI mappings: 70


,article_doi,dataset_doi,downloaded_pdf_path
1,10.1093/jhered/esx103,10.5061/dryad.121sk,fuster\10.1093_jhered_esx103.pdf
2,10.1371/journal.pone.0128238,10.5061/dryad.1771t,fuster\10.1371_journal.pone.0128238.pdf
3,10.1371/journal.pone.0073695,10.5061/dryad.1cc4v,fuster\10.1371_journal.pone.0073695.pdf
4,10.1002/ece3.4685,10.5061/dryad.24q6q70,fuster\10.1002_ece3.4685.pdf
5,10.1639/0007-2745-119.1.008,10.5061/dryad.24rj8,fuster\10.1639_0007-2745-119.1.008.pdf


In [17]:
# Load ground truth
gt_df = pd.read_excel(GROUND_TRUTH_PATH)
print(f"Ground truth: {len(gt_df)} total rows")

# Extract dataset DOI from URL column
gt_df['dataset_doi'] = gt_df['url'].str.replace('https://doi.org/', '', regex=False)

# Filter to DOIs that have downloaded PDFs
gt_matched = gt_df[gt_df['dataset_doi'].isin(doi_mapping.keys())].copy()
print(f"Ground truth with matched PDFs: {len(gt_matched)}")

# Add article DOI mapping
gt_matched['article_doi'] = gt_matched['dataset_doi'].map(doi_mapping)

gt_matched[['dataset_doi', 'article_doi', 'data_type', 'species']].head(5)

Ground truth: 418 total rows
Ground truth with matched PDFs: 70


,dataset_doi,article_doi,data_type,species
2,10.5061/dryad.121sk,10.1093/jhered/esx103,EBV genetic analysis,Glyptemys insculpta
4,10.5061/dryad.1771t,10.1371/journal.pone.0128238,density,"raccoons, striped skunks"
6,10.5061/dryad.1cc4v,10.1371/journal.pone.0073695,presence only,Rangifer tarandus caribou
7,10.5061/dryad.24q6q70,10.1002/ece3.4685,other,Rangifer tarandus caribou
8,10.5061/dryad.24rj8,10.1639/0007-2745-119.1.008,genetic analysis,Aspicilia bicensis


In [18]:
# Validate ground truth with DatasetFeaturesNormalized
RELEVANT_COLS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f',
    'species', 'referred_dataset'
]

ground_truth_by_dataset = {}
validation_errors = []

for _, row in gt_matched.iterrows():
    dataset_doi = row['dataset_doi']
    try:
        # Extract relevant columns and convert to dict
        row_dict = {col: row[col] for col in RELEVANT_COLS if col in row.index}
        
        # Validate using DatasetFeaturesNormalized
        validated = DatasetFeaturesNormalized.model_validate(row_dict)
        ground_truth_by_dataset[dataset_doi] = validated
    except Exception as e:
        validation_errors.append({'doi': dataset_doi, 'error': str(e)})

print(f"Successfully validated: {len(ground_truth_by_dataset)} records")
print(f"Validation errors: {len(validation_errors)} records")

if validation_errors:
    print("\nFirst 3 errors:")
    for err in validation_errors[:3]:
        print(f"  {err['doi']}: {err['error'][:100]}...")

Successfully validated: 70 records
Validation errors: 0 records


## Step 2: Configure Section Pipeline

Set up section-based extraction with selective section inclusion, gpt-5-mini, and the new section-aware system prompt.

In [19]:
# Pipeline configuration using section_pipeline
config = SectionClassificationConfig(
    model="gpt-5-mini",
    reasoning={"effort": "low"},
    max_output_tokens=4096,
    text_format=DatasetFeatures,
    system_message=SECTION_SYSTEM_MESSAGE,  # Section-aware prompt
    section_config=SectionSelectionConfig(
        include_all=True,  # Use ALL sections from PDFs for comprehensive extraction
        include_abstract=True,
    ),
    grobid_url="http://localhost:8070",
    pdf_dir=PDF_DIR,
    output_dir=OUTPUT_DIR,
)

print("Section Pipeline Configuration:")
print(f"  Model: {config.model}")
print(f"  Reasoning: {config.reasoning}")
print(f"  Section selection: include_all={config.section_config.include_all}")
print(f"  GROBID URL: {config.grobid_url}")
print(f"\nSystem Prompt Preview (first 200 chars):")
print(f"  {config.system_message[:200]}...")

Section Pipeline Configuration:
  Model: gpt-5-mini
  Reasoning: {'effort': 'low'}
  Section selection: include_all=True
  GROBID URL: http://localhost:8070

System Prompt Preview (first 200 chars):
  You are EcodataGPT, a structured data extraction engine for scientific article analysis.

Goal: Extract biodiversity dataset features from the provided sections of a scientific article (Abstract, Meth...


## Step 3: Run Section-Based Extraction

Process all PDFs through GROBID parsing and GPT classification using the section pipeline.

In [20]:
# Build input records for PDFs with ground truth
input_records = []
for _, row in gt_matched.iterrows():
    dataset_doi = row['dataset_doi']
    article_doi = row['article_doi']
    
    # Find PDF path from manifest
    manifest_row = downloaded_df[downloaded_df['dataset_doi'] == dataset_doi]
    if manifest_row.empty:
        continue
    
    pdf_path = PDF_DIR / manifest_row.iloc[0]['downloaded_pdf_path']
    if not pdf_path.exists():
        # Try alternate path construction
        pdf_path = PDF_DIR.parent / manifest_row.iloc[0]['downloaded_pdf_path']
    
    if pdf_path.exists():
        # Use SectionInputRecord with metadata for tracking
        input_records.append(SectionInputRecord(
            id=dataset_doi,  # Use dataset DOI as ID
            pdf_path=str(pdf_path),
            metadata={
                "article_doi": article_doi,
                "dataset_doi": dataset_doi,
            }
        ))

print(f"Input records to process: {len(input_records)}")

Input records to process: 70


In [21]:
# Process PDFs using SECTION PIPELINE
# Uses ThreadPoolTaskRunner for parallel processing

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_manifest_path = OUTPUT_DIR / f"section_results_{timestamp}.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running section-based pipeline on {len(input_records)} PDFs...")
print(f"Using {config.model} with {config.reasoning}")
print(f"System prompt: {config.system_message[:80]}...")
print()

# Run the Prefect flow with section-based extraction
results = section_classification_flow(
    input_records=input_records,
    config=config,
    output_manifest=output_manifest_path,
)

# Summary
success_count = sum(1 for r in results if r.status == "success")
error_count = sum(1 for r in results if r.status == "error")
print(f"\nProcessing complete: {success_count} success, {error_count} errors")
print(f"Results saved to: {output_manifest_path}")

Running section-based pipeline on 70 PDFs...
Using gpt-5-mini with {'effort': 'low'}
System prompt: You are EcodataGPT, a structured data extraction engine for scientific article a...



09:07:03.032 | INFO    | Flow run 'tourmaline-grebe' - Beginning flow run 'tourmaline-grebe' for flow 'section-classification-flow'

Processing 70 PDFs with GROBID + section selection...
  GROBID URL: http://localhost:8070
  Section types: ['ABSTRACT', 'METHODS']
GROBID parsing failed for data\pdfs\fuster\10.1639_0007-2745-119.1.008.pdf: GROBID processing failed: 500 Server Error: Server Error for url: http://localhost:8070/api/processFulltextDocument


09:07:04.868 | INFO    | Task run 'process_section_record-030' - Finished in state Completed()

GROBID parsing failed for data\pdfs\fuster\10.1111_mec.14361.pdf: GROBID processing failed: 500 Server Error: Server Error for url: http://localhost:8070/api/processFulltextDocument


...

09:11:30.063 | INFO    | Task run 'process_section_record-2f0' - Finished in state Completed()

09:11:31.794 | INFO    | Task run 'process_section_record-2c0' - Finished in state Completed()

Completed: 45 success, 25 failed
Average sections used: 14.1
Total cost: $0.4298
Saved output manifest to artifacts\section_results\section_results_20260113_090702.csv (70 records)


09:11:31.886 | INFO    | Flow run 'tourmaline-grebe' - Finished in state Completed()


Processing complete: 45 success, 25 errors
Results saved to: artifacts\section_results\section_results_20260113_090702.csv


## Step 4: Prepare Extractions for Evaluation

Convert extraction results to Pydantic models for evaluation.

In [22]:
# Build predictions by dataset DOI
predictions_by_dataset = {}

for result in results:
    if result.status != "success" or not result.extraction:
        continue
    
    # SectionOutputRecord uses 'id' field (which we set to dataset_doi)
    dataset_doi = result.id
    if not dataset_doi:
        continue
    
    try:
        # Create DatasetFeatures from extraction dict
        pred = DatasetFeatures.model_validate(result.extraction)
        predictions_by_dataset[dataset_doi] = pred
    except Exception as e:
        print(f"Error validating prediction for {dataset_doi}: {e}")

print(f"Valid predictions: {len(predictions_by_dataset)}")

# Find common DOIs between predictions and ground truth
common_dois = set(predictions_by_dataset.keys()) & set(ground_truth_by_dataset.keys())
print(f"Common DOIs for evaluation: {len(common_dois)}")

Valid predictions: 45
Common DOIs for evaluation: 45


## Step 5: Evaluate Section-Based Extraction

Compare section-based extractions against ground truth using evaluation framework.

In [23]:
# Evaluation fields
EVAL_FIELDS = [
    'data_type', 'geospatial_info_dataset', 'spatial_range_km2',
    'temporal_range', 'temp_range_i', 'temp_range_f', 'species'
]

# Evaluation configuration with fuzzy matching for species
eval_config = EvaluationConfig(
    casefold_strings=True,
    treat_lists_as_sets=True,
    fuzzy_match_fields={
        'species': FuzzyMatchConfig(threshold=80),
    }
)

# Filter to common DOIs
true_by_id = {doi: ground_truth_by_dataset[doi] for doi in common_dois}
pred_by_id = {doi: predictions_by_dataset[doi] for doi in common_dois}

# Run evaluation
section_report = evaluate_indexed(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=EVAL_FIELDS,
    config=eval_config,
)

print("SECTION-BASED Extraction Metrics:")
print("=" * 70)
display(section_report.metrics_to_pandas())

SECTION-BASED Extraction Metrics:


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
0,data_type,28,109,27,0,45,0.204380,0.509091,0.291667,0.622222,0.044444
1,geospatial_info_dataset,16,167,8,0,45,0.087432,0.666667,0.154589,0.355556,0.000000
2,spatial_range_km2,19,7,13,11,45,0.730769,0.593750,0.655172,0.666667,0.666667
6,species,19,217,51,0,45,0.080508,0.271429,0.124183,0.422222,0.044444
5,temp_range_f,26,12,11,6,45,0.684211,0.702703,0.693333,0.711111,0.711111
4,temp_range_i,33,5,4,6,45,0.868421,0.891892,0.880000,0.866667,0.866667
3,temporal_range,2,38,35,4,45,0.050000,0.054054,0.051948,0.133333,0.133333


In [24]:
# Aggregate metrics
section_micro = micro_average(section_report.field_metrics.values())
section_macro = macro_f1(section_report.field_metrics.values())

print("\nAggregate Metrics:")
print("=" * 50)
print(f"{'Metric':<25} {'Value':>15}")
print("-" * 50)
print(f"{'Micro Precision':<25} {section_micro.precision or 0:>15.3f}")
print(f"{'Micro Recall':<25} {section_micro.recall or 0:>15.3f}")
print(f"{'Micro F1':<25} {section_micro.f1 or 0:>15.3f}")
print(f"{'Macro F1':<25} {section_macro or 0:>15.3f}")
print(f"{'Records Evaluated':<25} {len(common_dois):>15}")
print("=" * 50)


Aggregate Metrics:
Metric                              Value
--------------------------------------------------
Micro Precision                     0.205
Micro Recall                        0.490
Micro F1                            0.289
Macro F1                            0.407
Records Evaluated                      45


## Step 6: Per-Field Analysis

In [25]:
# Per-field metrics for section-based extraction
field_data = []

for field in EVAL_FIELDS:
    fm = section_report.field_metrics.get(field)
    if fm:
        field_data.append({
            'field': field,
            'precision': round(fm.precision or 0, 3),
            'recall': round(fm.recall or 0, 3),
            'f1': round(fm.f1 or 0, 3),
            'tp': fm.tp,
            'fp': fm.fp,
            'fn': fm.fn,
            'exact_match_rate': round(fm.exact_match_rate or 0, 3),
        })

field_df = pd.DataFrame(field_data)
print("Per-Field Metrics:")
display(field_df)

# Identify best and worst performing fields
if len(field_data) > 0:
    best = max(field_data, key=lambda x: x['f1'])
    worst = min(field_data, key=lambda x: x['f1'])
    print(f"\nBest performing field: {best['field']} (F1={best['f1']})")
    print(f"Worst performing field: {worst['field']} (F1={worst['f1']})")

Per-Field Metrics:


,field,precision,recall,f1,tp,fp,fn,exact_match_rate
0,data_type,0.204,0.509,0.292,28,109,27,0.044
1,geospatial_info_dataset,0.087,0.667,0.155,16,167,8,0.000
2,spatial_range_km2,0.731,0.594,0.655,19,7,13,0.667
3,temporal_range,0.050,0.054,0.052,2,38,35,0.133
4,temp_range_i,0.868,0.892,0.880,33,5,4,0.867
5,temp_range_f,0.684,0.703,0.693,26,12,11,0.711
6,species,0.081,0.271,0.124,19,217,51,0.044



Best performing field: temp_range_i (F1=0.88)
Worst performing field: temporal_range (F1=0.052)


In [26]:
# Detailed per-record comparison results
print("Per-record, per-field results (mismatches highlighted):")
results_df = section_report.to_pandas()
mismatches = results_df[~results_df['match']]
print(f"\nTotal mismatches: {len(mismatches)}")
display(mismatches)

Per-record, per-field results (mismatches highlighted):

Total mismatches: 204


,record_id,field,true_value,pred_value,match,tp,fp,fn,tn
0,10.5061/dryad.121sk,data_type,[genetic_analysis],"[genetic_analysis, abundance]",False,1,1,0,0
1,10.5061/dryad.121sk,geospatial_info_dataset,None,"[site, geographic_features, administrative_units]",False,0,3,0,0
6,10.5061/dryad.121sk,species,[Glyptemys insculpta],[wood turtle (Glyptemys insculpta)],False,0,1,1,0
7,10.5061/dryad.1771t,data_type,[density],[abundance],False,0,1,1,0
8,10.5061/dryad.1771t,geospatial_info_dataset,[sample],"[sample, site, administrative_units, maps, geo...",False,1,4,0,0
...,...,...,...,...,...,...,...,...,...
302,10.5061/dryad.tc45g,geospatial_info_dataset,None,"[site, maps, administrative_units, distributio...",False,0,5,0,0
307,10.5061/dryad.tc45g,species,[Seiurus aurocapilla],"[Seiurus aurocapilla, ovenbird]",False,1,1,0,0
308,10.5061/dryad.xksn02vb9,data_type,[other],"[traits, genetic_analysis]",False,0,2,1,0
309,10.5061/dryad.xksn02vb9,geospatial_info_dataset,None,"[site, administrative_units]",False,0,2,0,0


## Step 7: Cost Analysis

Analyze token usage and costs for the section-based extraction pipeline.

In [27]:
# Cost analysis from results
successful_results = [r for r in results if r.status == "success"]

if successful_results:
    total_cost = sum(r.cost_usd or 0 for r in successful_results)
    total_input_tokens = sum(r.input_tokens or 0 for r in successful_results)
    total_output_tokens = sum(r.output_tokens or 0 for r in successful_results)
    total_fulltext_tokens = sum(r.fulltext_tokens or 0 for r in successful_results)
    total_abstract_tokens = sum(r.abstract_tokens or 0 for r in successful_results)
    total_sections = sum(r.sections_used or 0 for r in successful_results)
    
    print("\nCOST ANALYSIS")
    print("=" * 50)
    print(f"{'Metric':<30} {'Value':>18}")
    print("-" * 50)
    print(f"{'Total PDFs Processed':<30} {len(successful_results):>18}")
    print(f"{'Avg Sections Used':<30} {total_sections / len(successful_results):>18,.1f}")
    print(f"{'Avg Abstract Tokens':<30} {total_abstract_tokens / len(successful_results):>18,.0f}")
    print(f"{'Avg Full-text Tokens':<30} {total_fulltext_tokens / len(successful_results):>18,.0f}")
    print(f"{'Token Increase (avg)':<30} {(total_fulltext_tokens - total_abstract_tokens) / len(successful_results):>18,.0f}")
    print("-" * 50)
    print(f"{'Total Input Tokens':<30} {total_input_tokens:>18,}")
    print(f"{'Total Output Tokens':<30} {total_output_tokens:>18,}")
    print(f"{'Total Cost (USD)':<30} ${total_cost:>17.4f}")
    print(f"{'Avg Cost per PDF (USD)':<30} ${total_cost / len(successful_results):>17.5f}")
    print("=" * 50)


COST ANALYSIS
Metric                                      Value
--------------------------------------------------
Total PDFs Processed                           45
Avg Sections Used                            14.1
Avg Abstract Tokens                           315
Avg Full-text Tokens                        7,778
Token Increase (avg)                        7,463
--------------------------------------------------
Total Input Tokens                        411,513
Total Output Tokens                        87,235
Total Cost (USD)               $           0.4298
Avg Cost per PDF (USD)         $          0.00955
